In [1]:
import warnings
warnings.filterwarnings("ignore")

## Imports

In [2]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [3]:
from funcoes_escoragem import *

## Diretório

In [4]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [5]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [6]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        SAFE_CAST(JSON_EXTRACT_SCALAR(CR.request, '$.imovel.codigo_cidade_ibge') AS INT64) AS codigo_cidade_ibge,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 12 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,codigo_cidade_ibge,score_imobiliaria,renda_considerada,model,id_externo,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,44828300899,4492161,2026-08-11,2026-08-11,1,3501608,NI,0E-9,BLEND3_4,4492161,...,33,1786647574343,"{""imobiliaria_id"":""42097"",""imovel"":{""tipo"":""CO...",2026-08-13 19:38:51+00:00,1786649929005463084,44828300899,E,2026-08-13 19:42:36.201000+00:00,2026-08-11 06:27:06+00:00,REPROVAR
1,49412911823,4492180,2026-08-11,2026-08-11,1,3502804,NI,1370.000000000,BLEND3_4,4492180,...,31,1786647574343,"{""valor_aluguel"":700,""matchmaking_on"":false,""p...",2026-08-13 19:38:51+00:00,1786649929006801522,49412911823,A,2026-08-13 19:42:36.208000+00:00,2026-08-11 07:48:29+00:00,APROVAR
2,53226895915,4492195,2026-08-11,2026-08-11,1,<NA>,B,20344.500000000,BLEND_4,4492195,...,31,1786647574343,"{""principalDocument"":""53226895915"",""imobiliari...",2026-08-13 19:38:51+00:00,1786649929007575258,53226895915,A,2026-08-13 19:42:36.213000+00:00,2026-08-11 08:02:03+00:00,APROVAR
3,16775125631,4492221,2026-08-11,2026-08-11,1,3129806,D,1301.500000000,BLEND3_4,4492221,...,33,1786647574343,"{""valor_aluguel"":1100,""matchmaking_on"":false,""...",2026-08-13 19:38:51+00:00,1786649929008239360,16775125631,E,2026-08-13 19:42:36.223000+00:00,2026-08-11 08:10:48+00:00,REPROVAR
4,664989020,4477229,2026-08-11,2026-08-11,1,<NA>,D,6233.500000000,BLEND_4,4477229,...,32,1786647574343,"{""principalDocument"":""00664989020"",""imobiliari...",2026-08-13 19:38:51+00:00,1786649929008715722,00664989020,B,2026-08-13 19:42:36.224000+00:00,2026-08-11 08:12:10+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392266,41817763873,4887124,2026-11-02,2026-11-02,1,3509502,A,90899.500000000,BLEND_4,4887124,...,31,1786647574343,"{""principalDocument"":""41817763873"",""imobiliari...",2026-11-03 09:00:44+00:00,1793696441301116965,41817763873,A,2026-11-03 09:07:54.874000+00:00,2026-11-02 19:17:33+00:00,APROVAR
392267,41817763873,4887133,2026-11-02,2026-11-02,1,3509502,A,90899.500000000,BLEND_4,4887133,...,31,1786647574343,"{""principalDocument"":""41817763873"",""imobiliari...",2026-11-03 09:00:44+00:00,1793696441310253476,41817763873,A,2026-11-03 09:07:54.877000+00:00,2026-11-02 19:23:58+00:00,APROVAR
392268,83697381687,4887157,2026-11-02,2026-11-02,1,4202008,NI,8768.000000000,BLEND_4,4887157,...,31,1786647574343,"{""principalDocument"":""83697381687"",""imobiliari...",2026-11-03 09:00:44+00:00,1793696441378491910,83697381687,A,2026-11-03 09:07:54.889000+00:00,2026-11-02 20:33:01+00:00,APROVAR
392269,61721518355,4887161,2026-11-02,2026-11-02,1,4208203,NI,1370.000000000,BLEND_4,4887161,...,36,1786647574343,"{""principalDocument"":""61721518355"",""imobiliari...",2026-11-03 09:00:44+00:00,1793696441385932367,61721518355,D,2026-11-03 09:07:54.890000+00:00,2026-11-02 20:44:50+00:00,APROVAR


In [7]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

## Quebrar JSON - Retornado

In [8]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [9]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [10]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [11]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [12]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [13]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [14]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts


## Escoragem Blend4

In [15]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [16]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [17]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [18]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,codigo_cidade_ibge,score_imobiliaria,model,id_consulta,id_funcionalidade,...,request_modelEadVariables_vab_agropecuaria_cagr_2y,request_modelEadVariables_vab_industria_var_abs_1y,request_modelEadVariables_agency_segmentacao,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,44828300899,4492161,2026-08-11,2026-08-11,1,3501608,NI,BLEND3_4,6243537,33,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
1,49412911823,4492180,2026-08-11,2026-08-11,1,3502804,NI,BLEND3_4,6243561,31,...,NaN,NaN,NaN,1.0,854.0,CRED CARTAO,BLEND4,NaN,NaN,1370.0
2,53226895915,4492195,2026-08-11,2026-08-11,1,<NA>,B,BLEND_4,6243579,31,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,611.0,766.0,20344.5
3,16775125631,4492221,2026-08-11,2026-08-11,1,3129806,D,BLEND3_4,6243610,33,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1301.5
4,664989020,4477229,2026-08-11,2026-08-11,1,<NA>,D,BLEND_4,6243615,32,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,614.0,418.0,6233.5


In [19]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    358474
E_BVS      33797
dtype: int64

In [20]:
df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [21]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [22]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [23]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [24]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,codigo_cidade_ibge,score_imobiliaria,model,id_consulta,id_funcionalidade,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,44828300899,4492161,2026-08-11,2026-08-11,1,3501608,NI,BLEND3_4,6243537,33,...,0.000000,0.00,0.0,0.000000,0.000000,0.236499,0.544119,456.0,B,None
1,49412911823,4492180,2026-08-11,2026-08-11,1,3502804,NI,BLEND3_4,6243561,31,...,0.000000,-0.55,2.0,0.178930,0.000000,-0.495130,0.513276,487.0,A,B
2,53226895915,4492195,2026-08-11,2026-08-11,1,<NA>,B,BLEND_4,6243579,31,...,0.868805,1.45,0.0,-0.656552,-0.045977,-0.294017,0.315189,685.0,A,A
3,16775125631,4492221,2026-08-11,2026-08-11,1,3129806,D,BLEND3_4,6243610,33,...,0.000000,-0.80,0.0,0.917209,0.000000,0.787773,0.560404,440.0,C,E
4,664989020,4477229,2026-08-11,2026-08-11,1,<NA>,D,BLEND_4,6243615,32,...,-0.145773,0.35,2.0,-0.701652,0.026029,1.436102,0.531563,468.0,B,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392266,41817763873,4887124,2026-11-02,2026-11-02,1,3509502,A,BLEND_4,6732087,31,...,0.693878,-0.05,1.0,-0.912038,0.000000,-0.658138,0.241361,759.0,A,A
392267,41817763873,4887133,2026-11-02,2026-11-02,1,3509502,A,BLEND_4,6732096,31,...,0.693878,-0.05,1.0,-0.912038,0.000000,-0.658138,0.241361,759.0,A,A
392268,83697381687,4887157,2026-11-02,2026-11-02,1,4202008,NI,BLEND_4,6732130,31,...,0.705539,1.60,0.0,0.309932,0.000000,-0.835486,0.261989,738.0,A,A
392269,61721518355,4887161,2026-11-02,2026-11-02,1,4208203,NI,BLEND_4,6732134,36,...,-0.104956,-0.70,0.0,1.307563,0.000000,0.873699,0.605476,395.0,D,D


In [25]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                     22
BLEND3_4          98420
BLEND_4          275822
BVS_CUSTOM           95
BVS_CUSTOM_V2      4207
HFT1                829
HVA3              12321
HVA4                555
dtype: int64

In [26]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [27]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    358474
E_BVS      33797
dtype: int64

In [28]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                     22
BLEND3_4          98420
BLEND_4          275822
BVS_CUSTOM           95
BVS_CUSTOM_V2      4207
HFT1                829
HVA3              12321
HVA4                555
dtype: int64

In [29]:
cp_final_df.groupby("model", dropna=False).size()

model
                     22
BLEND3_4          98420
BLEND_4          275822
BVS_CUSTOM           95
BVS_CUSTOM_V2      4207
HFT1                829
HVA3              12321
HVA4                555
dtype: int64

In [30]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    358474
E_BVS      33797
dtype: int64

In [31]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

## PC Cidades e PC Imobiliária

In [32]:
project_id = 'loft-dl-datascience'

query_city = '''
select * from loft-dl-datascience.ds_risk_cp.blend3_2__variavel_cidade
'''

df_city = pd.read_gbq(query_city, project_id=project_id)
df_city.to_parquet(ANALYTICS_DIR/"df_city.parquet")

In [33]:
project_id = 'loft-dl-datascience'

query_agency = '''
select * from loft-dl-datascience.ds_risk_cp.blend3_2__variavel_imobiliaria
'''

df_agency = pd.read_gbq(query_agency, project_id=project_id)
df_agency.to_parquet(ANALYTICS_DIR/"df_agency.parquet")